
> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.

# Same-Stock vs. Cross-Stock Retrieval
## Expanded Current-S&P-500 Universe

This notebook evaluates the scalability of Future-Compatibility-Supervised Retrieval on roughly 500 current S&P 500 stocks using long-horizon Yahoo Finance data.

---

# Research questions

## Q1. Are cross-stock historical analogs useful?

We compare retrieval restricted to a query stock's own history (`SameStock`) with retrieval from the entire stock universe (`CrossStock`).

## Q2. Is cross-stock retrieval useful only because it provides more candidates?

We distinguish:

- SameStock Learned
- CrossStock Learned
- OtherStockOnly Learned

`OtherStockOnly` excludes candidates from the query ticker entirely.

## Q3. Does cross-stock memory improve coverage for short-history stocks?

Some current S&P 500 constituents have relatively short trading histories. Same-stock retrieval is unavailable when a query has fewer than Top-K eligible same-stock memories, whereas cross-stock retrieval can still operate. We therefore report common-query performance, same-stock coverage, and cross-stock all-query performance separately.

---

# Important limitation

The universe is based on a current-constituent snapshot and therefore has survivorship bias. This experiment tests **large-universe scalability and cross-security transfer**, not a survivorship-bias-free investment backtest.

---

# Fixed experimental setting

\[
L=20,\qquad H=5,\qquad K=10,\qquad M=100.
\]

The learned methods use three seeds, \(\{0,1,2\}\). Handcrafted alternatives select their retrieval hyperparameters using validation only.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


In [ ]:

from pathlib import Path
import copy
import math
import random
import warnings
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = REPO_WORK_ROOT / "finance_case"
RAW_DIR = DATA_ROOT / "raw"
RESULT_DIR = DATA_ROOT / "results" / "yahoo_crossstock_ablation"
MODEL_DIR = RESULT_DIR / "models"

RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = RAW_DIR / "yahoo_sp500_current_2000_2025.parquet"
VIX_FILE = RAW_DIR / "VIX_History.csv"

assert DATA_FILE.exists(), (
    f"Missing {DATA_FILE}\n"
    "Run stock_regime_yahoo_sp500_data_prep.ipynb first."
)
assert VIX_FILE.exists(), VIX_FILE

PAST_LEN = 20
PRED_LEN = 5
WINDOW_STRIDE = 5
LOCAL_HISTORY = 60

TOP_K = 10
CROSS_TOP_M = 100
SAME_TOP_M = 100
HAND_MAX_M = 200

SEEDS = [0, 1, 2]
# Final replication option:
# SEEDS = [0, 1, 2, 3, 4]

PRESELECT_BATCH = 256
TRAIN_BATCH = 128
EVAL_BATCH = 256

MAX_EPOCHS = 30
PATIENCE = 6
LR = 1e-3
WEIGHT_DECAY = 1e-4
TARGET_TEMP = 0.50

# Optional training cap for the first large-universe run.
# It preserves all validation/test queries but limits per-epoch training cost.
MAX_TRAIN_QUERIES_PER_PHASE = 60000

EPS = 1e-8

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:",
          torch.cuda.get_device_properties(0).total_memory / 1024**3)

print("DATA_FILE:", DATA_FILE)


## 1. Load expanded Yahoo universe

In [ ]:

data = pd.read_parquet(DATA_FILE)

data["Date"] = pd.to_datetime(
    data["Date"]
).dt.tz_localize(None)

data["Close"] = pd.to_numeric(
    data["Close"],
    errors="coerce",
)

data = (
    data.dropna(subset=["Ticker", "Date", "Close"])
    .sort_values(["Ticker", "Date"])
    .drop_duplicates(["Ticker", "Date"], keep="last")
    .reset_index(drop=True)
)

market_rows = data[data["Ticker"] == "SPY"].copy()
stocks = data[data["Ticker"] != "SPY"].copy()

print("Stock tickers:", stocks["Ticker"].nunique())
print("Rows:", len(stocks))
print("Range:", stocks["Date"].min(), "->", stocks["Date"].max())

display(
    stocks.groupby("Sector")["Ticker"]
    .nunique()
    .sort_values(ascending=False)
)


## 2. Build SPY + VIX global context

In [ ]:

spy = (
    market_rows
    .sort_values("Date")
    .set_index("Date")["Close"]
    .dropna()
)

vix_raw = pd.read_csv(VIX_FILE)
vix_raw.columns = [
    str(c).strip().upper()
    for c in vix_raw.columns
]

vix = vix_raw[["DATE", "CLOSE"]].copy()
vix.columns = ["Date", "VIX"]

vix["Date"] = pd.to_datetime(
    vix["Date"]
).dt.tz_localize(None)

vix["VIX"] = pd.to_numeric(
    vix["VIX"],
    errors="coerce",
)

vix = vix.dropna().sort_values("Date")

spy_log = np.log(spy)
spy_ret = spy_log.diff()

market = pd.DataFrame(index=spy.index)

market["MKT_MOM_5"] = spy_log - spy_log.shift(5)
market["MKT_MOM_20"] = spy_log - spy_log.shift(20)
market["MKT_MOM_60"] = spy_log - spy_log.shift(60)

market["MKT_RV_5"] = spy_ret.rolling(5).std() * np.sqrt(252)
market["MKT_RV_20"] = spy_ret.rolling(20).std() * np.sqrt(252)

market["MKT_VOL_RATIO"] = (
    market["MKT_RV_5"] /
    (market["MKT_RV_20"] + EPS)
)

market["MKT_DD_60"] = (
    spy_log -
    spy_log.rolling(60).max()
)

market = (
    market.reset_index()
    .merge(vix, on="Date", how="left")
    .set_index("Date")
)

market["VIX"] = market["VIX"].ffill(limit=3)
market["LOG_VIX"] = np.log(market["VIX"])

market["VIX_CHG_5"] = (
    market["LOG_VIX"] -
    market["LOG_VIX"].shift(5)
)

market["VIX_CHG_20"] = (
    market["LOG_VIX"] -
    market["LOG_VIX"].shift(20)
)

market["MKT_TREND_STRENGTH"] = (
    market["MKT_MOM_20"] /
    (
        market["MKT_RV_20"] *
        np.sqrt(20 / 252) +
        EPS
    )
).clip(-10, 10)

market = market.dropna()

GLOBAL_DYNAMIC = [
    "MKT_MOM_20",
    "MKT_MOM_60",
    "MKT_RV_20",
    "LOG_VIX",
    "MKT_MOM_5",
    "MKT_VOL_RATIO",
    "VIX_CHG_5",
    "VIX_CHG_20",
    "MKT_DD_60",
    "MKT_TREND_STRENGTH",
]

LOCAL_FEATURES = [
    "STK_RET_5",
    "STK_RET_20",
    "STK_RV_20",
    "STK_VOL_RATIO",
    "STK_DD_60",
    "REL_STRENGTH_20",
]

CONTEXT_COLS = (
    GLOBAL_DYNAMIC +
    LOCAL_FEATURES
)


## 3. Build stock windows

Compute log returns from adjusted Close for each ticker,

\[
r_t=\log P_t-\log P_{t-1},
\]

and construct the same past-pattern and future-target representation used by the finance retrieval experiments.


In [ ]:

ticker_meta = (
    stocks[
        [
            "Ticker",
            "Security",
            "Sector",
            "SubIndustry",
        ]
    ]
    .drop_duplicates("Ticker")
    .set_index("Ticker")
)

def make_windows_for_ticker(
    ticker,
    g,
    market_df,
):
    g = (
        g.sort_values("Date")
        .dropna(subset=["Close"])
        .reset_index(drop=True)
    )

    if len(g) < LOCAL_HISTORY + PRED_LEN + 100:
        return None

    dates = g["Date"].to_numpy()
    logp = np.log(
        g["Close"].to_numpy(dtype=np.float64)
    )

    rets = np.diff(
        logp,
        prepend=np.nan,
    )

    rows = []

    for t in range(
        max(PAST_LEN, LOCAL_HISTORY),
        len(g) - PRED_LEN,
        WINDOW_STRIDE,
    ):
        end_date = pd.Timestamp(dates[t])
        future_end_date = pd.Timestamp(
            dates[t + PRED_LEN]
        )

        if end_date not in market_df.index:
            continue

        # Avoid stitching across long missing periods.
        local_dates = pd.to_datetime(
            dates[
                t - LOCAL_HISTORY + 1:
                t + PRED_LEN + 1
            ]
        )

        gaps = (
            np.diff(local_dates.values)
            .astype("timedelta64[D]")
            .astype(int)
        )

        if len(gaps) and gaps.max() > 10:
            continue

        past_ret = rets[
            t - PAST_LEN + 1:
            t + 1
        ]

        future_ret = rets[
            t + 1:
            t + PRED_LEN + 1
        ]

        if not np.isfinite(past_ret).all():
            continue

        if not np.isfinite(future_ret).all():
            continue

        scale = np.std(past_ret) + EPS

        past_path = (
            np.cumsum(past_ret) /
            scale
        )

        norm = np.linalg.norm(past_path)

        if norm < EPS:
            continue

        pattern = (
            past_path / norm
        ).astype(np.float32)

        future_path = (
            np.cumsum(future_ret)
            .astype(np.float32)
        )

        ret5 = float(
            np.sum(rets[t-4:t+1])
        )

        ret20 = float(
            np.sum(past_ret)
        )

        rv5 = float(
            np.std(rets[t-4:t+1]) *
            np.sqrt(252)
        )

        rv20 = float(
            np.std(past_ret) *
            np.sqrt(252)
        )

        vol_ratio = (
            rv5 /
            (rv20 + EPS)
        )

        local60 = logp[t-59:t+1]

        dd60 = float(
            logp[t] -
            np.max(local60)
        )

        rel_strength20 = (
            ret20 -
            float(
                market_df.loc[
                    end_date,
                    "MKT_MOM_20",
                ]
            )
        )

        meta = ticker_meta.loc[ticker]

        row = {
            "Ticker": ticker,
            "Security": meta["Security"],
            "Sector": meta["Sector"],
            "SubIndustry": meta["SubIndustry"],

            "EndDate": end_date,
            "FutureEndDate": future_end_date,

            "Pattern": pattern,
            "FuturePath": future_path,
            "FutureTerminal": float(future_path[-1]),

            "STK_RET_5": ret5,
            "STK_RET_20": ret20,
            "STK_RV_20": rv20,
            "STK_VOL_RATIO": vol_ratio,
            "STK_DD_60": dd60,
            "REL_STRENGTH_20": rel_strength20,
        }

        for c in GLOBAL_DYNAMIC:
            row[c] = float(
                market_df.loc[
                    end_date,
                    c,
                ]
            )

        rows.append(row)

    if not rows:
        return None

    return pd.DataFrame(rows)

frames = []

groups = stocks.groupby(
    "Ticker",
    sort=False,
)

for ticker, g in tqdm(
    groups,
    total=stocks["Ticker"].nunique(),
    desc="Building stock windows",
):
    w = make_windows_for_ticker(
        ticker,
        g,
        market,
    )

    if w is not None:
        frames.append(w)

windows = pd.concat(
    frames,
    ignore_index=True,
)

print("Windows:", len(windows))
print("Tickers:", windows["Ticker"].nunique())
print("Range:", windows["EndDate"].min(), "->", windows["FutureEndDate"].max())

windows.to_parquet(
    RESULT_DIR / "00_expanded_windows.parquet",
    index=False,
)


## 4. Strict chronological splits

In [ ]:

TRAIN_MEMORY_CUTOFF = pd.Timestamp("2009-12-31")
TRAIN_START = pd.Timestamp("2010-01-01")
TRAIN_END = pd.Timestamp("2014-12-31")

VAL_MEMORY_CUTOFF = pd.Timestamp("2014-12-31")
VAL_START = pd.Timestamp("2015-01-01")
VAL_END = pd.Timestamp("2019-12-31")

TEST_MEMORY_CUTOFF = pd.Timestamp("2019-12-31")
TEST_START = pd.Timestamp("2020-01-01")

train_cand = windows[
    windows["FutureEndDate"] <=
    TRAIN_MEMORY_CUTOFF
].reset_index(drop=True)

train_query = windows[
    (windows["EndDate"] >= TRAIN_START) &
    (windows["FutureEndDate"] <= TRAIN_END)
].reset_index(drop=True)

val_cand = windows[
    windows["FutureEndDate"] <=
    VAL_MEMORY_CUTOFF
].reset_index(drop=True)

val_query = windows[
    (windows["EndDate"] >= VAL_START) &
    (windows["FutureEndDate"] <= VAL_END)
].reset_index(drop=True)

test_cand = windows[
    windows["FutureEndDate"] <=
    TEST_MEMORY_CUTOFF
].reset_index(drop=True)

test_query = windows[
    windows["EndDate"] >=
    TEST_START
].reset_index(drop=True)

split_table = pd.DataFrame({
    "Split": [
        "Train memory",
        "Train queries",
        "Validation memory",
        "Validation queries",
        "Test memory",
        "Test queries",
    ],

    "N": [
        len(train_cand),
        len(train_query),
        len(val_cand),
        len(val_query),
        len(test_cand),
        len(test_query),
    ],

    "Tickers": [
        train_cand["Ticker"].nunique(),
        train_query["Ticker"].nunique(),
        val_cand["Ticker"].nunique(),
        val_query["Ticker"].nunique(),
        test_cand["Ticker"].nunique(),
        test_query["Ticker"].nunique(),
    ],
})

display(split_table)

split_table.to_csv(
    RESULT_DIR / "01_split_summary.csv",
    index=False,
)


## 5. Tensor and scaling helpers

In [ ]:

def stack_col(df, col):
    return np.stack(
        df[col].to_numpy()
    ).astype(np.float32)

def basic_tensors(cand_df, query_df):
    return {
        "C_PATTERN": torch.tensor(
            stack_col(cand_df, "Pattern"),
            dtype=torch.float32,
            device=DEVICE,
        ),
        "C_FUTURE": torch.tensor(
            stack_col(cand_df, "FuturePath"),
            dtype=torch.float32,
            device=DEVICE,
        ),
        "Q_PATTERN": torch.tensor(
            stack_col(query_df, "Pattern"),
            dtype=torch.float32,
            device=DEVICE,
        ),
        "Q_FUTURE": torch.tensor(
            stack_col(query_df, "FuturePath"),
            dtype=torch.float32,
            device=DEVICE,
        ),
    }

def fit_robust_scaler(df, cols):
    x = df[cols].to_numpy(dtype=np.float32)

    med = np.nanmedian(x, axis=0)
    q25 = np.nanpercentile(x, 25, axis=0)
    q75 = np.nanpercentile(x, 75, axis=0)

    iqr = q75 - q25
    iqr = np.where(
        iqr < 1e-6,
        1.0,
        iqr,
    )

    return (
        med.astype(np.float32),
        iqr.astype(np.float32),
    )

def transform_context(
    df,
    cols,
    med,
    iqr,
    clip=8.0,
):
    x = df[cols].to_numpy(dtype=np.float32)

    x = (
        x - med
    ) / iqr

    x = np.clip(
        x,
        -clip,
        clip,
    )

    return x.astype(np.float32)

train_basic = basic_tensors(
    train_cand,
    train_query,
)

val_basic = basic_tensors(
    val_cand,
    val_query,
)

test_basic = basic_tensors(
    test_cand,
    test_query,
)


## 6. Cross-stock Pattern Top-200

For a several-hundred-stock universe, the candidate-search matrix multiplication is the dominant retrieval computation. It is evaluated in batches on the GPU.


In [ ]:

@torch.no_grad()
def cross_pattern_preselect(
    q_pattern,
    c_pattern,
    top_m,
    batch_size=256,
):
    all_idx = []
    all_score = []

    for start in tqdm(
        range(
            0,
            len(q_pattern),
            batch_size,
        ),
        desc=f"Cross-stock Top-{top_m}",
    ):
        end = min(
            start + batch_size,
            len(q_pattern),
        )

        score = (
            q_pattern[start:end] @
            c_pattern.T
        )

        top_score, top_idx = torch.topk(
            score,
            k=min(
                top_m,
                c_pattern.shape[0],
            ),
            dim=1,
            largest=True,
        )

        all_idx.append(
            top_idx.cpu()
        )

        all_score.append(
            top_score.cpu()
        )

    return (
        torch.cat(all_idx),
        torch.cat(all_score),
    )

train_cross_idx200, train_cross_score200 = (
    cross_pattern_preselect(
        train_basic["Q_PATTERN"],
        train_basic["C_PATTERN"],
        HAND_MAX_M,
        PRESELECT_BATCH,
    )
)

val_cross_idx200, val_cross_score200 = (
    cross_pattern_preselect(
        val_basic["Q_PATTERN"],
        val_basic["C_PATTERN"],
        HAND_MAX_M,
        PRESELECT_BATCH,
    )
)

test_cross_idx200, test_cross_score200 = (
    cross_pattern_preselect(
        test_basic["Q_PATTERN"],
        test_basic["C_PATTERN"],
        HAND_MAX_M,
        PRESELECT_BATCH,
    )
)

torch.save(
    {
        "train_idx": train_cross_idx200,
        "train_score": train_cross_score200,
        "val_idx": val_cross_idx200,
        "val_score": val_cross_score200,
        "test_idx": test_cross_idx200,
        "test_score": test_cross_score200,
    },
    RESULT_DIR / "02_cross_pattern_preselection.pt",
)


## 7. Same-stock Pattern Top-100 and coverage

In [ ]:

def same_stock_preselect(
    cand_df,
    query_df,
    c_pattern,
    q_pattern,
    top_m,
):
    cand_groups = {
        str(t): np.asarray(idx, dtype=np.int64)
        for t, idx in cand_df.groupby("Ticker").indices.items()
    }

    query_groups = query_df.groupby("Ticker").indices

    all_idx = np.full(
        (len(query_df), top_m),
        -1,
        dtype=np.int64,
    )

    all_score = np.full(
        (len(query_df), top_m),
        -np.inf,
        dtype=np.float32,
    )

    counts = np.zeros(
        len(query_df),
        dtype=np.int32,
    )

    for ticker, qids in tqdm(
        query_groups.items(),
        total=len(query_groups),
        desc=f"Same-stock Top-{top_m}",
    ):
        ticker = str(ticker)
        qids = np.asarray(qids, dtype=np.int64)

        cids = cand_groups.get(ticker)

        if cids is None or len(cids) == 0:
            continue

        k = min(
            top_m,
            len(cids),
        )

        score = (
            q_pattern[qids] @
            c_pattern[cids].T
        )

        top_score, local_idx = torch.topk(
            score,
            k=k,
            dim=1,
            largest=True,
        )

        cids_tensor = torch.tensor(
            cids,
            dtype=torch.long,
            device=DEVICE,
        )

        global_idx = cids_tensor[
            local_idx
        ]

        all_idx[
            qids,
            :k
        ] = global_idx.cpu().numpy()

        all_score[
            qids,
            :k
        ] = top_score.cpu().numpy()

        counts[qids] = len(cids)

    return (
        torch.tensor(
            all_idx,
            dtype=torch.long,
        ),
        torch.tensor(
            all_score,
            dtype=torch.float32,
        ),
        counts,
    )

train_same_idx, train_same_score, train_same_count = (
    same_stock_preselect(
        train_cand,
        train_query,
        train_basic["C_PATTERN"],
        train_basic["Q_PATTERN"],
        SAME_TOP_M,
    )
)

val_same_idx, val_same_score, val_same_count = (
    same_stock_preselect(
        val_cand,
        val_query,
        val_basic["C_PATTERN"],
        val_basic["Q_PATTERN"],
        SAME_TOP_M,
    )
)

test_same_idx, test_same_score, test_same_count = (
    same_stock_preselect(
        test_cand,
        test_query,
        test_basic["C_PATTERN"],
        test_basic["Q_PATTERN"],
        SAME_TOP_M,
    )
)

train_same_mask = (
    train_same_count >= TOP_K
)

val_same_mask = (
    val_same_count >= TOP_K
)

test_same_mask = (
    test_same_count >= TOP_K
)

coverage = pd.DataFrame({
    "Split": [
        "Train",
        "Validation",
        "Test",
    ],

    "NQueries": [
        len(train_query),
        len(val_query),
        len(test_query),
    ],

    "SameStockEligible": [
        int(train_same_mask.sum()),
        int(val_same_mask.sum()),
        int(test_same_mask.sum()),
    ],
})

coverage["Coverage"] = (
    coverage["SameStockEligible"] /
    coverage["NQueries"]
)

display(coverage)

coverage.to_csv(
    RESULT_DIR / "03_same_stock_coverage.csv",
    index=False,
)


## 8. Forecast metrics

In [ ]:

@torch.no_grad()
def metrics_from_indices(
    idx_cpu,
    q_future,
    c_future,
):
    idx = idx_cpu.to(DEVICE)

    retrieved = c_future[idx]
    true = q_future

    analog_mse = (
        (retrieved - true[:, None, :]) ** 2
    ).mean(dim=(1, 2))

    pred = retrieved.mean(dim=1)

    forecast_mse = (
        (pred - true) ** 2
    ).mean(dim=1)

    terminal_mae = (
        pred[:, -1] -
        true[:, -1]
    ).abs()

    direction = (
        torch.sign(pred[:, -1]) ==
        torch.sign(true[:, -1])
    ).float()

    return pd.DataFrame({
        "AnalogFutureMSE":
            analog_mse.cpu().numpy(),

        "ForecastMSE":
            forecast_mse.cpu().numpy(),

        "TerminalMAE":
            terminal_mae.cpu().numpy(),

        "DirectionCorrect":
            direction.cpu().numpy(),
    })

def summarize(m):
    return {
        "AnalogFutureMSE":
            float(m["AnalogFutureMSE"].mean()),

        "ForecastMSE":
            float(m["ForecastMSE"].mean()),

        "TerminalMAE":
            float(m["TerminalMAE"].mean()),

        "DirectionAcc":
            float(m["DirectionCorrect"].mean()),
    }


## 9. Handcrafted reranker and fair validation tuning

In [ ]:

def context_tensors_expanding(
    cand_df,
    query_df,
):
    med, iqr = fit_robust_scaler(
        cand_df,
        CONTEXT_COLS,
    )

    c = torch.tensor(
        transform_context(
            cand_df,
            CONTEXT_COLS,
            med,
            iqr,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    q = torch.tensor(
        transform_context(
            query_df,
            CONTEXT_COLS,
            med,
            iqr,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    return c, q

@torch.no_grad()
def handcrafted_rerank(
    pre_idx_cpu,
    pre_score_cpu,
    q_context,
    c_context,
    query_mask,
    M,
    lam,
    tau,
    top_k=10,
    batch_size=256,
):
    qids_all = np.where(
        query_mask
    )[0]

    output_idx = []

    for start in range(
        0,
        len(qids_all),
        batch_size,
    ):
        qids = qids_all[
            start:
            start + batch_size
        ]

        idx_cpu = pre_idx_cpu[
            qids,
            :M
        ]

        pscore_cpu = pre_score_cpu[
            qids,
            :M
        ]

        valid = (
            idx_cpu >= 0
        )

        safe_idx = (
            idx_cpu
            .clamp_min(0)
            .to(DEVICE)
        )

        pscore = (
            pscore_cpu
            .to(DEVICE)
        )

        pscore = pscore.masked_fill(
            ~valid.to(DEVICE),
            -torch.inf,
        )

        qctx = (
            q_context[qids]
            .unsqueeze(1)
        )

        cctx = c_context[
            safe_idx
        ]

        dist2 = (
            (qctx - cctx) ** 2
        ).mean(dim=2)

        cscore = torch.exp(
            -0.5 *
            dist2 /
            (tau ** 2)
        )

        total = (
            pscore +
            lam * cscore
        )

        local = torch.topk(
            total,
            k=top_k,
            dim=1,
            largest=True,
        ).indices

        global_idx = torch.gather(
            safe_idx,
            1,
            local,
        )

        output_idx.append(
            global_idx.cpu()
        )

    return (
        qids_all,
        torch.cat(output_idx),
    )

val_cctx, val_qctx = (
    context_tensors_expanding(
        val_cand,
        val_query,
    )
)

test_cctx, test_qctx = (
    context_tensors_expanding(
        test_cand,
        test_query,
    )
)

M_GRID = [25, 50, 100, 200]
LAMBDA_GRID = [
    0.05,
    0.10,
    0.20,
    0.40,
]
TAU_GRID = [
    0.50,
    1.00,
    2.00,
]

def tune_handcrafted(
    name,
    pre_idx,
    pre_score,
    mask,
):
    rows = []

    for M in M_GRID:
        M_actual = min(
            M,
            pre_idx.shape[1],
        )

        for lam in LAMBDA_GRID:
            for tau in TAU_GRID:

                qids, idx = (
                    handcrafted_rerank(
                        pre_idx,
                        pre_score,
                        val_qctx,
                        val_cctx,
                        mask,
                        M=M_actual,
                        lam=lam,
                        tau=tau,
                        top_k=TOP_K,
                    )
                )

                m = metrics_from_indices(
                    idx,
                    val_basic["Q_FUTURE"][qids],
                    val_basic["C_FUTURE"],
                )

                row = {
                    "Pool":
                        name,

                    "M":
                        M_actual,

                    "Lambda":
                        lam,

                    "Tau":
                        tau,
                }

                row.update(
                    summarize(m)
                )

                rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "ForecastMSE",
                "AnalogFutureMSE",
            ]
        )
        .reset_index(drop=True)
    )

same_hand_grid = tune_handcrafted(
    "SameStock",
    val_same_idx,
    val_same_score,
    val_same_mask,
)

cross_hand_grid = tune_handcrafted(
    "CrossStock",
    val_cross_idx200,
    val_cross_score200,
    np.ones(
        len(val_query),
        dtype=bool,
    ),
)

hand_grid = pd.concat(
    [
        same_hand_grid,
        cross_hand_grid,
    ],
    ignore_index=True,
)

display(
    hand_grid.groupby("Pool")
    .head(10)
)

hand_grid.to_csv(
    RESULT_DIR / "04_handcrafted_validation_grid.csv",
    index=False,
)


## 10. Learned future-compatibility reranker

In [ ]:

class FutureCompatibilityReranker(
    nn.Module
):
    def __init__(
        self,
        context_dim,
        hidden_dim=128,
        dropout=0.10,
        initial_alpha=0.10,
    ):
        super().__init__()

        input_dim = (
            1 +
            4 * context_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                hidden_dim // 2,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim // 2,
                1,
            ),
        )

        raw = math.log(
            math.exp(initial_alpha) -
            1.0
        )

        self.log_alpha_raw = nn.Parameter(
            torch.tensor(
                raw,
                dtype=torch.float32,
            )
        )

    @property
    def alpha(self):
        return F.softplus(
            self.log_alpha_raw
        )

    def forward(
        self,
        pattern_score,
        query_context,
        cand_context,
    ):
        B, M, D = (
            cand_context.shape
        )

        q = (
            query_context[:, None, :]
            .expand(-1, M, -1)
        )

        signed_diff = (
            q -
            cand_context
        )

        abs_diff = (
            signed_diff.abs()
        )

        feat = torch.cat(
            [
                pattern_score[..., None],
                q,
                cand_context,
                signed_diff,
                abs_diff,
            ],
            dim=-1,
        )

        delta = (
            self.mlp(feat)
            .squeeze(-1)
        )

        return (
            pattern_score +
            self.alpha *
            delta
        )

def listwise_loss(
    score,
    future_dist,
    valid,
):
    score = score.masked_fill(
        ~valid,
        -torch.inf,
    )

    fd = future_dist.masked_fill(
        ~valid,
        torch.nan,
    )

    mean = torch.nanmean(
        fd,
        dim=1,
        keepdim=True,
    )

    centered = (
        fd -
        mean
    )

    var = torch.nanmean(
        centered ** 2,
        dim=1,
        keepdim=True,
    )

    std = torch.sqrt(
        var
    ).clamp_min(1e-6)

    z = (
        future_dist -
        mean
    ) / std

    target_logits = (
        -z /
        TARGET_TEMP
    )

    target_logits = (
        target_logits
        .masked_fill(
            ~valid,
            -torch.inf,
        )
    )

    target = torch.softmax(
        target_logits,
        dim=1,
    )

    log_prob = F.log_softmax(
        score,
        dim=1,
    )

    # Avoid 0 * -inf on padded candidates.
    per_item = torch.where(
        valid,
        target * log_prob,
        torch.zeros_like(log_prob),
    )

    return -(
        per_item.sum(dim=1)
        .mean()
    )


## 11. Learned context coordinate system

In [ ]:

learn_med, learn_iqr = (
    fit_robust_scaler(
        train_cand,
        CONTEXT_COLS,
    )
)

def add_learn_context(
    basic,
    cand_df,
    query_df,
):
    out = dict(basic)

    out["C_CONTEXT"] = torch.tensor(
        transform_context(
            cand_df,
            CONTEXT_COLS,
            learn_med,
            learn_iqr,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    out["Q_CONTEXT"] = torch.tensor(
        transform_context(
            query_df,
            CONTEXT_COLS,
            learn_med,
            learn_iqr,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    return out

train_t = add_learn_context(
    train_basic,
    train_cand,
    train_query,
)

val_t = add_learn_context(
    val_basic,
    val_cand,
    val_query,
)

test_t = add_learn_context(
    test_basic,
    test_cand,
    test_query,
)

train_cross_idx = (
    train_cross_idx200[
        :, :CROSS_TOP_M
    ]
)

train_cross_score = (
    train_cross_score200[
        :, :CROSS_TOP_M
    ]
)

val_cross_idx = (
    val_cross_idx200[
        :, :CROSS_TOP_M
    ]
)

val_cross_score = (
    val_cross_score200[
        :, :CROSS_TOP_M
    ]
)

test_cross_idx = (
    test_cross_idx200[
        :, :CROSS_TOP_M
    ]
)

test_cross_score = (
    test_cross_score200[
        :, :CROSS_TOP_M
    ]
)


## 12. Training-query subsampling for the large universe

Validation and test use all eligible queries. To control training cost, each training phase uses at most 60,000 uniformly sampled queries; the sample is seed-dependent. If fewer than 60,000 queries are available, all are used.


In [ ]:

def sample_training_qids(
    mask,
    max_n,
    seed,
):
    qids = np.where(mask)[0]

    if len(qids) <= max_n:
        return qids

    rng = np.random.default_rng(
        seed
    )

    return np.sort(
        rng.choice(
            qids,
            size=max_n,
            replace=False,
        )
    )


## 13. Learned training/evaluation helpers

In [ ]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def prepare_batch(
    qids_cpu,
    pre_idx_cpu,
    pre_score_cpu,
    phase_t,
):
    idx_cpu = pre_idx_cpu[
        qids_cpu
    ]

    score_cpu = pre_score_cpu[
        qids_cpu
    ]

    valid = (
        idx_cpu >= 0
    )

    safe_idx = (
        idx_cpu
        .clamp_min(0)
        .to(DEVICE)
    )

    pscore = (
        score_cpu
        .to(DEVICE)
    )

    pscore = pscore.masked_fill(
        ~valid.to(DEVICE),
        -torch.inf,
    )

    qids = torch.tensor(
        qids_cpu,
        dtype=torch.long,
        device=DEVICE,
    )

    qctx = (
        phase_t[
            "Q_CONTEXT"
        ][qids]
    )

    cctx = (
        phase_t[
            "C_CONTEXT"
        ][safe_idx]
    )

    qfuture = (
        phase_t[
            "Q_FUTURE"
        ][qids]
    )

    cfuture = (
        phase_t[
            "C_FUTURE"
        ][safe_idx]
    )

    future_dist = (
        (
            cfuture -
            qfuture[:, None, :]
        ) ** 2
    ).mean(
        dim=2
    )

    return (
        safe_idx,
        pscore,
        qctx,
        cctx,
        future_dist,
        valid.to(DEVICE),
    )

@torch.no_grad()
def evaluate_pool(
    model,
    phase_t,
    pre_idx,
    pre_score,
    query_mask,
):
    model.eval()

    qids_all = np.where(
        query_mask
    )[0]

    all_idx = []

    for start in range(
        0,
        len(qids_all),
        EVAL_BATCH,
    ):
        qids = qids_all[
            start:
            start + EVAL_BATCH
        ]

        (
            safe_idx,
            pscore,
            qctx,
            cctx,
            _,
            valid,
        ) = prepare_batch(
            qids,
            pre_idx,
            pre_score,
            phase_t,
        )

        score = model(
            pscore,
            qctx,
            cctx,
        )

        score = score.masked_fill(
            ~valid,
            -torch.inf,
        )

        local = torch.topk(
            score,
            k=TOP_K,
            dim=1,
            largest=True,
        ).indices

        global_idx = torch.gather(
            safe_idx,
            1,
            local,
        )

        all_idx.append(
            global_idx.cpu()
        )

    idx = torch.cat(
        all_idx
    )

    m = metrics_from_indices(
        idx,
        phase_t["Q_FUTURE"][qids_all],
        phase_t["C_FUTURE"],
    )

    return (
        qids_all,
        idx,
        m,
    )

def train_phase_a(
    seed,
    train_t,
    val_t,
    train_idx,
    train_score,
    val_idx,
    val_score,
    train_mask,
    val_mask,
):
    set_seed(seed)

    model = (
        FutureCompatibilityReranker(
            context_dim=len(
                CONTEXT_COLS
            ),
            hidden_dim=128,
            dropout=0.10,
        )
        .to(DEVICE)
    )

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    train_qids = (
        sample_training_qids(
            train_mask,
            MAX_TRAIN_QUERIES_PER_PHASE,
            seed=1000 + seed,
        )
    )

    best_epoch = None
    best_val_mse = float("inf")
    wait = 0

    history = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        model.train()

        perm = np.random.permutation(
            train_qids
        )

        losses = []

        for start in range(
            0,
            len(perm),
            TRAIN_BATCH,
        ):
            qids = perm[
                start:
                start + TRAIN_BATCH
            ]

            (
                _,
                pscore,
                qctx,
                cctx,
                future_dist,
                valid,
            ) = prepare_batch(
                qids,
                train_idx,
                train_score,
                train_t,
            )

            opt.zero_grad(
                set_to_none=True
            )

            score = model(
                pscore,
                qctx,
                cctx,
            )

            loss = listwise_loss(
                score,
                future_dist,
                valid,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            opt.step()

            losses.append(
                loss.item()
            )

        _, _, val_m = evaluate_pool(
            model,
            val_t,
            val_idx,
            val_score,
            val_mask,
        )

        val_summary = summarize(
            val_m
        )

        val_mse = (
            val_summary[
                "ForecastMSE"
            ]
        )

        history.append({
            "Seed":
                seed,

            "Epoch":
                epoch,

            "TrainN":
                len(train_qids),

            "TrainLoss":
                float(np.mean(losses)),

            "Alpha":
                float(model.alpha.item()),

            "ValForecastMSE":
                val_mse,

            "ValAnalogFutureMSE":
                val_summary[
                    "AnalogFutureMSE"
                ],
        })

        if (
            val_mse <
            best_val_mse - 1e-9
        ):
            best_val_mse = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        if wait >= PATIENCE:
            break

    return (
        best_epoch,
        best_val_mse,
        pd.DataFrame(history),
    )

def refit_model(
    seed,
    epochs,
    train_t,
    val_t,
    train_idx,
    train_score,
    val_idx,
    val_score,
    train_mask,
    val_mask,
):
    set_seed(seed)

    model = (
        FutureCompatibilityReranker(
            context_dim=len(
                CONTEXT_COLS
            ),
            hidden_dim=128,
            dropout=0.10,
        )
        .to(DEVICE)
    )

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    phase_specs = [
        (
            train_t,
            train_idx,
            train_score,
            train_mask,
            2000 + seed,
        ),
        (
            val_t,
            val_idx,
            val_score,
            val_mask,
            3000 + seed,
        ),
    ]

    for epoch in range(
        1,
        epochs + 1,
    ):
        order = [0, 1]

        random.Random(
            seed * 1000 +
            epoch
        ).shuffle(order)

        model.train()

        for phase_id in order:
            (
                pt,
                pidx,
                pscore_all,
                pmask,
                sample_seed,
            ) = phase_specs[
                phase_id
            ]

            qids_all = (
                sample_training_qids(
                    pmask,
                    MAX_TRAIN_QUERIES_PER_PHASE,
                    seed=(
                        sample_seed +
                        epoch
                    ),
                )
            )

            perm = np.random.permutation(
                qids_all
            )

            for start in range(
                0,
                len(perm),
                TRAIN_BATCH,
            ):
                qids = perm[
                    start:
                    start +
                    TRAIN_BATCH
                ]

                (
                    _,
                    pscore,
                    qctx,
                    cctx,
                    future_dist,
                    valid,
                ) = prepare_batch(
                    qids,
                    pidx,
                    pscore_all,
                    pt,
                )

                opt.zero_grad(
                    set_to_none=True
                )

                score = model(
                    pscore,
                    qctx,
                    cctx,
                )

                loss = listwise_loss(
                    score,
                    future_dist,
                    valid,
                )

                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0,
                )

                opt.step()

    return model


## 14. Train SameStock and CrossStock models

In [ ]:

pool_specs = {
    "SameStock": {
        "train_idx":
            train_same_idx,

        "train_score":
            train_same_score,

        "val_idx":
            val_same_idx,

        "val_score":
            val_same_score,

        "test_idx":
            test_same_idx,

        "test_score":
            test_same_score,

        "train_mask":
            train_same_mask,

        "val_mask":
            val_same_mask,

        "test_mask":
            test_same_mask,
    },

    "CrossStock": {
        "train_idx":
            train_cross_idx,

        "train_score":
            train_cross_score,

        "val_idx":
            val_cross_idx,

        "val_score":
            val_cross_score,

        "test_idx":
            test_cross_idx,

        "test_score":
            test_cross_score,

        "train_mask":
            np.ones(
                len(train_query),
                dtype=bool,
            ),

        "val_mask":
            np.ones(
                len(val_query),
                dtype=bool,
            ),

        "test_mask":
            np.ones(
                len(test_query),
                dtype=bool,
            ),
    },
}

learned_results = {}
learned_indices = {}
models = {}
history_frames = []

for pool_name, spec in pool_specs.items():

    for seed in SEEDS:

        print(
            "\n",
            "=" * 80
        )
        print(
            pool_name,
            "| seed",
            seed
        )
        print(
            "=" * 80
        )

        best_epoch, best_val, hist = (
            train_phase_a(
                seed,
                train_t,
                val_t,
                spec["train_idx"],
                spec["train_score"],
                spec["val_idx"],
                spec["val_score"],
                spec["train_mask"],
                spec["val_mask"],
            )
        )

        hist["Pool"] = (
            pool_name
        )

        history_frames.append(
            hist
        )

        print(
            "Best epoch:",
            best_epoch,
            "Best val MSE:",
            best_val,
        )

        model = refit_model(
            seed,
            best_epoch,
            train_t,
            val_t,
            spec["train_idx"],
            spec["train_score"],
            spec["val_idx"],
            spec["val_score"],
            spec["train_mask"],
            spec["val_mask"],
        )

        qids, idx, m = evaluate_pool(
            model,
            test_t,
            spec["test_idx"],
            spec["test_score"],
            spec["test_mask"],
        )

        key = (
            pool_name,
            seed,
        )

        learned_results[key] = {
            "qids":
                qids,

            "metrics":
                m,

            "best_epoch":
                best_epoch,

            "best_val":
                best_val,
        }

        learned_indices[
            key
        ] = idx

        models[
            key
        ] = model

        torch.save(
            {
                "Pool":
                    pool_name,

                "Seed":
                    seed,

                "BestEpoch":
                    best_epoch,

                "StateDict":
                    model.state_dict(),

                "ContextCols":
                    CONTEXT_COLS,

                "Median":
                    learn_med,

                "IQR":
                    learn_iqr,
            },
            MODEL_DIR /
            f"{pool_name}_seed{seed}.pt",
        )

history_table = pd.concat(
    history_frames,
    ignore_index=True,
)

history_table.to_csv(
    RESULT_DIR / "05_training_history.csv",
    index=False,
)


## 15. Common-query Pattern and Handcrafted baselines

In [ ]:

common_qids = np.where(
    test_same_mask
)[0]

# Pattern
same_pattern_idx = (
    test_same_idx[
        common_qids,
        :TOP_K
    ]
)

cross_pattern_idx = (
    test_cross_idx[
        common_qids,
        :TOP_K
    ]
)

same_pattern_m = metrics_from_indices(
    same_pattern_idx,
    test_basic["Q_FUTURE"][common_qids],
    test_basic["C_FUTURE"],
)

cross_pattern_m = metrics_from_indices(
    cross_pattern_idx,
    test_basic["Q_FUTURE"][common_qids],
    test_basic["C_FUTURE"],
)

# Handcrafted
same_best = (
    same_hand_grid.iloc[0]
)

cross_best = (
    cross_hand_grid.iloc[0]
)

same_qids, same_hand_idx = (
    handcrafted_rerank(
        test_same_idx,
        test_same_score,
        test_qctx,
        test_cctx,
        test_same_mask,
        M=int(
            same_best["M"]
        ),
        lam=float(
            same_best["Lambda"]
        ),
        tau=float(
            same_best["Tau"]
        ),
        top_k=TOP_K,
    )
)

all_mask = np.ones(
    len(test_query),
    dtype=bool,
)

cross_all_qids, cross_hand_idx_all = (
    handcrafted_rerank(
        test_cross_idx200,
        test_cross_score200,
        test_qctx,
        test_cctx,
        all_mask,
        M=int(
            cross_best["M"]
        ),
        lam=float(
            cross_best["Lambda"]
        ),
        tau=float(
            cross_best["Tau"]
        ),
        top_k=TOP_K,
    )
)

# cross_all_qids is 0..N-1
cross_hand_idx_common = (
    cross_hand_idx_all[
        common_qids
    ]
)

same_hand_m = metrics_from_indices(
    same_hand_idx,
    test_basic["Q_FUTURE"][common_qids],
    test_basic["C_FUTURE"],
)

cross_hand_m = metrics_from_indices(
    cross_hand_idx_common,
    test_basic["Q_FUTURE"][common_qids],
    test_basic["C_FUTURE"],
)


## 16. Learned seed-average on common queries

In [ ]:

def learned_seed_average_common(
    pool_name,
):
    arrays = {
        metric: []
        for metric in [
            "AnalogFutureMSE",
            "ForecastMSE",
            "TerminalMAE",
            "DirectionCorrect",
        ]
    }

    seed_rows = []

    for seed in SEEDS:
        r = learned_results[
            (
                pool_name,
                seed,
            )
        ]

        qids = r[
            "qids"
        ]

        pos = {
            int(q): i
            for i, q in enumerate(qids)
        }

        rows = [
            pos[int(q)]
            for q in common_qids
        ]

        m = (
            r["metrics"]
            .iloc[rows]
            .reset_index(drop=True)
        )

        s = summarize(m)

        seed_rows.append({
            "Pool":
                pool_name,

            "Seed":
                seed,

            "BestEpoch":
                r["best_epoch"],

            **s,
        })

        for metric in arrays:
            arrays[
                metric
            ].append(
                m[metric]
                .to_numpy()
            )

    mean_df = pd.DataFrame({
        metric:
            np.stack(
                values,
                axis=0,
            ).mean(
                axis=0
            )
        for metric, values
        in arrays.items()
    })

    return (
        mean_df,
        pd.DataFrame(
            seed_rows
        ),
    )

same_learned_mean, same_seed_table = (
    learned_seed_average_common(
        "SameStock"
    )
)

cross_learned_mean, cross_seed_table = (
    learned_seed_average_common(
        "CrossStock"
    )
)

seed_table = pd.concat(
    [
        same_seed_table,
        cross_seed_table,
    ],
    ignore_index=True,
)

display(seed_table)

seed_table.to_csv(
    RESULT_DIR / "06_learned_seed_results_common.csv",
    index=False,
)


## 17. Other-stock-only learned diagnostic

In [ ]:

@torch.no_grad()
def other_stock_only_eval(
    model,
    qids_eval,
):
    c_ticker = np.asarray(
        test_cand[
            "Ticker"
        ].astype(str)
    )

    q_ticker = np.asarray(
        test_query[
            "Ticker"
        ].astype(str)
    )

    all_idx = []

    for start in range(
        0,
        len(qids_eval),
        EVAL_BATCH,
    ):
        qids = qids_eval[
            start:
            start + EVAL_BATCH
        ]

        idx_cpu = (
            test_cross_idx[
                qids
            ]
        )

        pscore = (
            test_cross_score[
                qids
            ]
            .to(DEVICE)
        )

        idx = idx_cpu.to(
            DEVICE
        )

        qctx = (
            test_t[
                "Q_CONTEXT"
            ][qids]
        )

        cctx = (
            test_t[
                "C_CONTEXT"
            ][idx]
        )

        score = model(
            pscore,
            qctx,
            cctx,
        )

        retrieved_ticker = (
            c_ticker[
                idx_cpu.numpy()
            ]
        )

        same = (
            retrieved_ticker ==
            q_ticker[qids, None]
        )

        same_t = torch.tensor(
            same,
            dtype=torch.bool,
            device=DEVICE,
        )

        available = (
            (~same_t)
            .sum(dim=1)
        )

        if (
            available <
            TOP_K
        ).any():
            raise RuntimeError(
                "Not enough other-stock candidates. "
                "Increase CROSS_TOP_M."
            )

        score = score.masked_fill(
            same_t,
            -torch.inf,
        )

        local = torch.topk(
            score,
            k=TOP_K,
            dim=1,
            largest=True,
        ).indices

        global_idx = torch.gather(
            idx,
            1,
            local,
        )

        all_idx.append(
            global_idx.cpu()
        )

    idx = torch.cat(
        all_idx
    )

    m = metrics_from_indices(
        idx,
        test_t["Q_FUTURE"][qids_eval],
        test_t["C_FUTURE"],
    )

    return idx, m

other_metrics_by_seed = []

for seed in SEEDS:
    _, m = other_stock_only_eval(
        models[
            (
                "CrossStock",
                seed,
            )
        ],
        common_qids,
    )

    other_metrics_by_seed.append(
        m
    )

other_mean = pd.DataFrame({
    metric:
        np.stack(
            [
                m[metric].to_numpy()
                for m in other_metrics_by_seed
            ],
            axis=0,
        ).mean(axis=0)
    for metric in [
        "AnalogFutureMSE",
        "ForecastMSE",
        "TerminalMAE",
        "DirectionCorrect",
    ]
})

display(
    pd.DataFrame({
        "OtherStockOnly_Learned":
            summarize(other_mean)
    }).T
)


## 18. How much does unrestricted learned retrieval actually use other stocks?

In [ ]:

cand_ticker = (
    test_cand[
        "Ticker"
    ].astype(str)
    .to_numpy()
)

query_ticker = (
    test_query[
        "Ticker"
    ].astype(str)
    .to_numpy()
)

fraction_rows = []

for seed in SEEDS:
    key = (
        "CrossStock",
        seed,
    )

    idx = (
        learned_indices[
            key
        ].numpy()
    )

    qids = (
        learned_results[
            key
        ]["qids"]
    )

    retrieved = (
        cand_ticker[
            idx
        ]
    )

    qname = (
        query_ticker[
            qids
        ][:, None]
    )

    frac = (
        retrieved !=
        qname
    ).mean(axis=1)

    fraction_rows.append({
        "Seed":
            seed,

        "MeanOtherStockFraction@K":
            float(
                frac.mean()
            ),

        "MedianOtherStockFraction@K":
            float(
                np.median(frac)
            ),

        "PctQueriesWithAnyOtherStock":
            float(
                (frac > 0)
                .mean()
            ),

        "PctQueriesAllOtherStocks":
            float(
                (frac == 1)
                .mean()
            ),
    })

cross_fraction = pd.DataFrame(
    fraction_rows
)

display(cross_fraction)

cross_fraction.to_csv(
    RESULT_DIR / "07_cross_stock_fraction.csv",
    index=False,
)


## 19. Same-stock vs cross-stock Future Oracle

In [ ]:

@torch.no_grad()
def oracle_from_pool(
    pre_idx,
    qids,
    q_future,
    c_future,
):
    all_idx = []

    for start in range(
        0,
        len(qids),
        EVAL_BATCH,
    ):
        qb = qids[
            start:
            start + EVAL_BATCH
        ]

        idx_cpu = (
            pre_idx[qb]
        )

        valid = (
            idx_cpu >= 0
        )

        idx = (
            idx_cpu
            .clamp_min(0)
            .to(DEVICE)
        )

        cf = c_future[
            idx
        ]

        qf = q_future[
            qb
        ]

        dist = (
            (
                cf -
                qf[:, None, :]
            ) ** 2
        ).mean(
            dim=2
        )

        dist = dist.masked_fill(
            ~valid.to(DEVICE),
            torch.inf,
        )

        local = torch.topk(
            dist,
            k=TOP_K,
            dim=1,
            largest=False,
        ).indices

        global_idx = torch.gather(
            idx,
            1,
            local,
        )

        all_idx.append(
            global_idx.cpu()
        )

    idx = torch.cat(
        all_idx
    )

    m = metrics_from_indices(
        idx,
        q_future[qids],
        c_future,
    )

    return idx, m

same_oracle_idx, same_oracle_m = (
    oracle_from_pool(
        test_same_idx,
        common_qids,
        test_basic["Q_FUTURE"],
        test_basic["C_FUTURE"],
    )
)

cross_oracle_idx, cross_oracle_m = (
    oracle_from_pool(
        test_cross_idx,
        common_qids,
        test_basic["Q_FUTURE"],
        test_basic["C_FUTURE"],
    )
)


## 20. Main paper-style common-query comparison

In [ ]:

main_table = pd.DataFrame({
    "SameStock_Pattern":
        summarize(
            same_pattern_m
        ),

    "SameStock_Handcrafted":
        summarize(
            same_hand_m
        ),

    "SameStock_Learned":
        summarize(
            same_learned_mean
        ),

    "CrossStock_Pattern":
        summarize(
            cross_pattern_m
        ),

    "CrossStock_Handcrafted":
        summarize(
            cross_hand_m
        ),

    "CrossStock_Learned":
        summarize(
            cross_learned_mean
        ),

    "OtherStockOnly_Learned":
        summarize(
            other_mean
        ),

    "SameStock_FutureOracle":
        summarize(
            same_oracle_m
        ),

    "CrossStock_FutureOracle":
        summarize(
            cross_oracle_m
        ),
}).T

display(main_table)

main_table.to_csv(
    RESULT_DIR / "08_main_common_query_comparison.csv"
)


## 21. CrossStock learned on all eligible test queries

In [ ]:

all_cross_rows = []

for seed in SEEDS:
    r = learned_results[
        (
            "CrossStock",
            seed,
        )
    ]

    row = {
        "Seed":
            seed,

        "BestEpoch":
            r[
                "best_epoch"
            ],
    }

    row.update(
        summarize(
            r["metrics"]
        )
    )

    all_cross_rows.append(
        row
    )

all_cross_table = pd.DataFrame(
    all_cross_rows
)

display(all_cross_table)

all_cross_table.to_csv(
    RESULT_DIR / "09_crossstock_all_query_results.csv",
    index=False,
)


## 22. Date-aware bootstrap: CrossStock Learned vs. SameStock Learned

Compare seed-averaged query-level losses on the common-query subset. Positive

\[
L_{\mathrm{same}}-L_{\mathrm{cross}}>0
\]

means CrossStock is better. The moving-block bootstrap is performed over dates to account for temporal dependence.


In [ ]:

def date_level_diff(
    dates,
    baseline,
    proposed,
):
    df = pd.DataFrame({
        "Date":
            pd.to_datetime(
                dates
            ),

        "Diff":
            np.asarray(
                baseline
            ) -
            np.asarray(
                proposed
            ),
    })

    return (
        df.groupby(
            "Date"
        )["Diff"]
        .mean()
        .sort_index()
    )

def cluster_bootstrap(
    x,
    n_boot=5000,
    seed=42,
):
    a = x.to_numpy(
        dtype=np.float64
    )

    rng = (
        np.random.default_rng(
            seed
        )
    )

    n = len(a)

    boot = np.empty(
        n_boot
    )

    for b in range(
        n_boot
    ):
        idx = rng.integers(
            0,
            n,
            size=n,
        )

        boot[b] = (
            a[idx].mean()
        )

    return {
        "ObservedImprovement":
            a.mean(),

        "CI_2.5%":
            np.quantile(
                boot,
                0.025,
            ),

        "CI_97.5%":
            np.quantile(
                boot,
                0.975,
            ),

        "P(improvement>0)":
            np.mean(
                boot > 0
            ),

        "N_dates":
            n,
    }

def moving_block_bootstrap(
    x,
    block_len=20,
    n_boot=5000,
    seed=42,
):
    a = x.to_numpy(
        dtype=np.float64
    )

    n = len(a)

    rng = np.random.default_rng(
        seed
    )

    n_blocks = math.ceil(
        n /
        block_len
    )

    max_start = (
        n -
        block_len
    )

    boot = np.empty(
        n_boot
    )

    for b in range(
        n_boot
    ):
        pieces = []

        for _ in range(
            n_blocks
        ):
            s = rng.integers(
                0,
                max_start + 1,
            )

            pieces.append(
                a[
                    s:
                    s + block_len
                ]
            )

        sample = (
            np.concatenate(
                pieces
            )[:n]
        )

        boot[b] = (
            sample.mean()
        )

    return {
        "ObservedImprovement":
            a.mean(),

        "CI_2.5%":
            np.quantile(
                boot,
                0.025,
            ),

        "CI_97.5%":
            np.quantile(
                boot,
                0.975,
            ),

        "P(improvement>0)":
            np.mean(
                boot > 0
            ),

        "N_dates":
            n,

        "BlockLength":
            block_len,
    }

dates_common = (
    test_query.iloc[
        common_qids
    ]["EndDate"]
    .to_numpy()
)

boot_rows = []

for metric in [
    "AnalogFutureMSE",
    "ForecastMSE",
    "TerminalMAE",
]:
    d = date_level_diff(
        dates_common,
        same_learned_mean[
            metric
        ],
        cross_learned_mean[
            metric
        ],
    )

    c = cluster_bootstrap(
        d,
        seed=100,
    )

    c["Metric"] = metric
    c["Bootstrap"] = "DateCluster"
    boot_rows.append(c)

    b = moving_block_bootstrap(
        d,
        block_len=20,
        seed=200,
    )

    b["Metric"] = metric
    b["Bootstrap"] = "MovingBlock"
    boot_rows.append(b)

bootstrap_table = pd.DataFrame(
    boot_rows
)

display(bootstrap_table)

bootstrap_table.to_csv(
    RESULT_DIR / "10_cross_vs_same_bootstrap.csv",
    index=False,
)


## 23. Sector analysis of cross-stock retrieval

In [ ]:

# Use seed-average cross-stock metrics on common queries.
sector_info = (
    test_query.iloc[
        common_qids
    ][
        [
            "Ticker",
            "Sector",
            "EndDate",
        ]
    ]
    .reset_index(drop=True)
)

sector_info["SameForecastMSE"] = (
    same_learned_mean[
        "ForecastMSE"
    ].to_numpy()
)

sector_info["CrossForecastMSE"] = (
    cross_learned_mean[
        "ForecastMSE"
    ].to_numpy()
)

sector_rows = []

for sector, g in sector_info.groupby(
    "Sector",
    dropna=False,
):
    same_mse = (
        g[
            "SameForecastMSE"
        ].mean()
    )

    cross_mse = (
        g[
            "CrossForecastMSE"
        ].mean()
    )

    sector_rows.append({
        "Sector":
            sector,

        "N":
            len(g),

        "SameStockMSE":
            same_mse,

        "CrossStockMSE":
            cross_mse,

        "CrossImprove_%":
            100 * (
                same_mse -
                cross_mse
            ) /
            same_mse,
    })

sector_table = (
    pd.DataFrame(
        sector_rows
    )
    .sort_values(
        "CrossImprove_%",
        ascending=False,
    )
)

display(sector_table)

sector_table.to_csv(
    RESULT_DIR / "11_sector_crossstock_analysis.csv",
    index=False,
)


## 24. Final decision summary

In [ ]:

same_mse = (
    main_table.loc[
        "SameStock_Learned",
        "ForecastMSE",
    ]
)

cross_mse = (
    main_table.loc[
        "CrossStock_Learned",
        "ForecastMSE",
    ]
)

other_mse = (
    main_table.loc[
        "OtherStockOnly_Learned",
        "ForecastMSE",
    ]
)

same_oracle_mse = (
    main_table.loc[
        "SameStock_FutureOracle",
        "ForecastMSE",
    ]
)

cross_oracle_mse = (
    main_table.loc[
        "CrossStock_FutureOracle",
        "ForecastMSE",
    ]
)

moving = (
    bootstrap_table[
        (
            bootstrap_table[
                "Metric"
            ] ==
            "ForecastMSE"
        ) &
        (
            bootstrap_table[
                "Bootstrap"
            ] ==
            "MovingBlock"
        )
    ]
    .iloc[0]
)

decision = pd.DataFrame({
    "Question": [
        "Cross learned beats same learned",
        "Cross-vs-same moving-block CI > 0",
        "Cross oracle beats same oracle",
        "Other-only MSE / Cross MSE",
        "Same-stock test coverage",
        "Mean other-stock fraction@K",
    ],

    "Value": [
        bool(
            cross_mse <
            same_mse
        ),

        bool(
            moving[
                "CI_2.5%"
            ] > 0
        ),

        bool(
            cross_oracle_mse <
            same_oracle_mse
        ),

        float(
            other_mse /
            cross_mse
        ),

        float(
            test_same_mask.mean()
        ),

        float(
            cross_fraction[
                "MeanOtherStockFraction@K"
            ].mean()
        ),
    ],
})

display(decision)

decision.to_csv(
    RESULT_DIR / "12_decision_summary.csv",
    index=False,
)


# Interpretation Guide

Default output directory:

```text
_work/finance_case/results/yahoo_crossstock_ablation/
```

Key files:

```text
03_same_stock_coverage.csv
06_learned_seed_results_common.csv
07_cross_stock_fraction.csv
08_main_common_query_comparison.csv
09_crossstock_all_query_results.csv
10_cross_vs_same_bootstrap.csv
11_sector_crossstock_analysis.csv
12_decision_summary.csv
```

## Case A — CrossStock Learned beats SameStock Learned

If CrossStock has lower loss on common queries and the moving-block confidence interval is positive, this supports transfer of predictive historical analogs across securities.

## Case B — OtherStockOnly remains strong

If performance remains competitive after completely excluding the query ticker, historical relevance is not purely security-specific.

## Case C — Similar common-query accuracy but higher cross-stock coverage

Cross-stock memory can still provide a practical cold-start mechanism for stocks with limited own-history memory.

## Case D — SameStock is stronger

Unrestricted cross-stock retrieval is adding noise; this would motivate more structured candidate policies rather than stronger claims about cross-stock transfer.

## Paper limitation

Because the universe is a current S&P 500 snapshot, results are used only as evidence for scalability, cross-stock transfer, and large-universe robustness. They are not presented as a survivorship-bias-free historical index backtest.
